# Notebook 26 — Great Britain race-population completeness

## Audit question

**Is the Great Britain race population in immutable Source Version 1 materially compromised in 2020, with 2019 and 2021 serving as controls?**

This notebook is a database/source-correctness investigation discovered while Great Britain Study 05 was examining sex restrictions. Study 05 remains paused at its pushed checkpoint while this audit is resolved.

### Working hypothesis

> **H₁: the Source Version 1 Great Britain race-population defect is concentrated in 2020.**

This is a hypothesis to test, not a conclusion.

The investigation therefore starts with **2019 / 2020 / 2021**, using 2019 and 2021 as controls. It does **not** begin with a uniform 2015–2026 audit.

If the controls are clean and 2020 is anomalous, the next stage will localise the 2020 discrepancy by month and racecourse. Additional surrounding years will then be used as falsification controls.


## Evidence boundary and standing rules

- **British Horseracing Authority (BHA) data is the primary source for the official British race population.**
- Immutable Source Version 1 and accepted Database v4 remain read-only.
- The prohibited **Source Version 1 off-time field must not be queried, selected, loaded, displayed, sorted on, derived from or used for reconciliation anywhere in this investigation.**
- When individual-race reconciliation becomes necessary, use governed/project-owned Inside Rails identities and governed racecourse identity.
- Fixture/date/course race counts are screening evidence only. A count mismatch is a **candidate population discrepancy** until investigated.
- A BHA fixture record is not itself evidence that races were run: the BHA fixture endpoint can return records whose race-list endpoint contains zero races. Official race counts must therefore come from the fixture race-list records.
- Every substantive result must be followed immediately by a written finding before moving to the next analytical question.
- This notebook must fail closed rather than silently switching source/database artefacts.

### Known positive control entering this audit

The Study 05 checkpoint already established the following defect for **6 June 2020**:

| Source | Newcastle | Newmarket | Lingfield Park | GB races |
|---|---:|---:|---:|---:|
| Official BHA race lists | 10 | 9 | 9 | 28 |
| Immutable Source Version 1 | 10 | 0 | 0 | 10 |
| Database v4 | 10 | 0 | 0 | 10 |

Source Version 1 also contains two South African races on that date.

The upstream tracing is already complete: the 18 Newmarket/Lingfield races are absent from Source Version 1 itself, and Database v4 inherited that omission.

Notebook 26 uses this date as a **positive-control sanity check** before testing the wider 2019/2020/2021 hypothesis.


In [1]:
from pathlib import Path
import hashlib
import json
import re
import time
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not locate Inside Rails project root")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


PROJECT_ROOT = find_project_root(Path.cwd())

SOURCE_VERSION_1 = (
    PROJECT_ROOT
    / "data/raw/form_2015-present/form_2015-present/raceform.db"
)

DATABASE_V4 = (
    PROJECT_ROOT
    / "data/processed/database/releases/inside_rails_v4.sqlite3"
)

EXPECTED_SOURCE_V1_SHA256 = (
    "77b5dbbbfdee69d4d92a582655344e1e5ba29ca4646a5999c383de8161eeeaa7"
)

EXPECTED_DATABASE_V4_SHA256 = (
    "45ad0c3d81d457385d655d9c47b030c5815c638e477281a9be8aabf164eecff7"
)

assert SOURCE_VERSION_1.is_file(), (
    f"Accepted Source Version 1 not found: {SOURCE_VERSION_1}"
)

assert DATABASE_V4.is_file(), (
    f"Accepted Database v4 not found: {DATABASE_V4}"
)

source_v1_sha256 = sha256_file(SOURCE_VERSION_1)
database_v4_sha256 = sha256_file(DATABASE_V4)

assert source_v1_sha256 == EXPECTED_SOURCE_V1_SHA256, (
    "Source Version 1 SHA-256 mismatch: "
    f"expected {EXPECTED_SOURCE_V1_SHA256}, observed {source_v1_sha256}"
)

assert database_v4_sha256 == EXPECTED_DATABASE_V4_SHA256, (
    "Database v4 SHA-256 mismatch: "
    f"expected {EXPECTED_DATABASE_V4_SHA256}, observed {database_v4_sha256}"
)

print("Project root:", PROJECT_ROOT)
print("Source Version 1:", SOURCE_VERSION_1)
print("Source Version 1 SHA-256:", source_v1_sha256)
print("Database v4:", DATABASE_V4)
print("Database v4 SHA-256:", database_v4_sha256)


Project root: /home/rob/Documents/inside-rails-horse-racing
Source Version 1: /home/rob/Documents/inside-rails-horse-racing/data/raw/form_2015-present/form_2015-present/raceform.db
Source Version 1 SHA-256: 77b5dbbbfdee69d4d92a582655344e1e5ba29ca4646a5999c383de8161eeeaa7
Database v4: /home/rob/Documents/inside-rails-horse-racing/data/processed/database/releases/inside_rails_v4.sqlite3
Database v4 SHA-256: 45ad0c3d81d457385d655d9c47b030c5815c638e477281a9be8aabf164eecff7


### Execution checkpoint — exact artefact identity

Do not continue unless the preceding cell completes without assertion failure and prints the documented SHA-256 identities for both immutable artefacts.

No Source Version 1 table or field is queried by that gate; it verifies only the file identity.

For the initial population screen, Database v4's governed GB race-occurrence view will be used as the source-backed comparator because it preserves one row per admitted GB source race occurrence while supplying project-owned race occurrence and racecourse identities. Direct Source Version 1 inspection is unnecessary for the first screening stage and would add no independent population information because the known defect has already been traced upstream.

## Next question

Can the audit reproduce the known **6 June 2020** BHA-versus-source-backed population discrepancy without using the prohibited source off-time field?


In [2]:
# Acquire the current public BHA frontend Authorization value.
#
# This reuses the acquisition method established in Study 05.
# The value is held in memory only and is never printed or persisted.

BHA_API_ROOT = "https://api09.horseracing.software"

BHA_APP_JS = (
    "https://www.britishhorseracing.com/"
    "wp-content/themes/bha/library/js/angular/app.js"
)

USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/131.0 Safari/537.36"
)

app_request = Request(
    BHA_APP_JS,
    headers={"User-Agent": USER_AGENT},
)

with urlopen(app_request, timeout=30) as response:
    app_js_text = response.read().decode("utf-8", errors="replace")

authorization_pattern = re.compile(
    r"""
    \$httpProvider
    \.defaults
    \.headers
    \.common
    \[['"]Authorization['"]\]
    \s*=\s*
    ['"]([^'"]+)['"]
    """,
    re.VERBOSE,
)

authorization_matches = []

for line in app_js_text.splitlines():
    stripped = line.lstrip()

    if stripped.startswith("//"):
        continue

    match = authorization_pattern.search(line)

    if match:
        authorization_matches.append(match.group(1))

assert len(authorization_matches) == 1, (
    "Expected exactly one active BHA Authorization assignment; "
    f"found {len(authorization_matches)}."
)

authorization_value = authorization_matches[0]


def get_bha_json(url: str) -> dict:
    request = Request(
        url,
        headers={
            "Authorization": authorization_value,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": "https://www.britishhorseracing.com/",
            "User-Agent": USER_AGENT,
        },
    )

    with urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


print("BHA acquisition helper ready.")


BHA acquisition helper ready.


In [3]:
# Positive control: official BHA race population for 6 June 2020.
#
# Important:
# - fixture rows are not counted as races;
# - every fixture's race-list endpoint is inspected;
# - zero-race fixture records remain visible in the screening output;
# - no race-time field is required for this population count.

POSITIVE_CONTROL_DATE = "2020-06-06"

fixture_params = {
    "fromdate": POSITIVE_CONTROL_DATE,
    "todate": POSITIVE_CONTROL_DATE,
    "resultsAvailable": 1,
    "order": "asc",
    "page": 1,
    "per_page": 100,
}

fixture_url = (
    f"{BHA_API_ROOT}/bha/v1/fixtures/?"
    + urlencode(fixture_params)
)

fixture_payload = get_bha_json(fixture_url)
bha_fixtures = fixture_payload["data"]

bha_fixture_race_counts = []
bha_race_rows = []

for fixture in bha_fixtures:
    race_list_url = (
        f"{BHA_API_ROOT}/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/"
        f"{fixture['fixtureId']}/races"
    )

    race_payload = get_bha_json(race_list_url)
    races = race_payload["data"]

    bha_fixture_race_counts.append(
        {
            "fixture_year": fixture.get("fixtureYear"),
            "fixture_id": fixture.get("fixtureId"),
            "course_name": fixture.get("courseName"),
            "races": len(races),
        }
    )

    for race in races:
        bha_race_rows.append(
            {
                "race_date": race.get("raceDate"),
                "course_name": fixture.get("courseName"),
                "fixture_id": fixture.get("fixtureId"),
                "bha_race_id": race.get("raceId"),
                "division_sequence": race.get("divisionSequence"),
                "race_name": race.get("raceName"),
            }
        )

    # Retain the gentle pacing used during Study 05 to avoid unnecessary
    # pressure on the public BHA service.
    time.sleep(1.5)

bha_fixture_race_counts = pd.DataFrame(bha_fixture_race_counts)
bha_races = pd.DataFrame(bha_race_rows)

bha_active_fixture_counts = (
    bha_fixture_race_counts
    .loc[bha_fixture_race_counts["races"] > 0, ["course_name", "races"]]
    .sort_values("course_name")
    .reset_index(drop=True)
)

expected_bha_counts = {
    "Lingfield Park": 9,
    "Newcastle": 10,
    "Newmarket": 9,
}

observed_bha_counts = dict(
    zip(
        bha_active_fixture_counts["course_name"],
        bha_active_fixture_counts["races"],
    )
)

assert observed_bha_counts == expected_bha_counts, (
    "Positive-control BHA course counts changed: "
    f"expected {expected_bha_counts}, observed {observed_bha_counts}"
)

assert len(bha_races) == 28, (
    f"Expected 28 official BHA races, observed {len(bha_races)}"
)

print("BHA fixture records returned:", len(bha_fixture_race_counts))
print("BHA race records returned:", len(bha_races))
print()
print("BHA race counts by fixture record:")
display(
    bha_fixture_race_counts[
        ["course_name", "fixture_id", "races"]
    ]
)
print()
print("BHA active race counts:")
display(bha_active_fixture_counts)


BHA fixture records returned: 9
BHA race records returned: 28

BHA race counts by fixture record:


,course_name,fixture_id,races
0,Newcastle,19006,10
1,Newmarket,19002,9
2,Doncaster,718,0
3,Hexham,931,0
4,Musselburgh,12730,0
5,Worcester,1833,0
6,Epsom Downs,766,0
7,Lingfield Park,19014,9
8,Chepstow,658,0



BHA active race counts:


,course_name,races
0,Lingfield Park,9
1,Newcastle,10
2,Newmarket,9


In [4]:
# Positive control: source-backed Database v4 population on 6 June 2020.
#
# This deliberately selects only project-owned/governed fields needed for the
# population check. The prohibited Source Version 1 off-time field is neither
# queried nor exposed.
#
# We inspect:
#   1. the governed GB race-occurrence view;
#   2. the general reconciled race-occurrence view to confirm the two non-GB
#      source races already known from Study 05.

with connect_read_only(DATABASE_V4) as connection:

    v4_gb_positive_control = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            governed_racecourse_name
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = ?
        ORDER BY
            governed_racecourse_name,
            source_race_occurrence_code
        """,
        connection,
        params=[POSITIVE_CONTROL_DATE],
    )

    v4_all_positive_control = pd.read_sql_query(
        """
        SELECT
            source_race_occurrence_code,
            raw_date AS race_date,
            candidate_jurisdiction,
            candidate_course_label
        FROM view_reconciled_race_occurrences
        WHERE raw_date = ?
        ORDER BY
            candidate_jurisdiction,
            candidate_course_label,
            source_race_occurrence_code
        """,
        connection,
        params=[POSITIVE_CONTROL_DATE],
    )

v4_gb_course_counts = (
    v4_gb_positive_control
    .groupby("governed_racecourse_name", as_index=False)
    .size()
    .rename(columns={"size": "races"})
)

v4_jurisdiction_counts = (
    v4_all_positive_control
    .groupby("candidate_jurisdiction", as_index=False)
    .size()
    .rename(columns={"size": "races"})
)

assert len(v4_gb_positive_control) == 10, (
    f"Expected 10 source-backed GB races, observed {len(v4_gb_positive_control)}"
)

assert v4_gb_course_counts.to_dict("records") == [
    {"governed_racecourse_name": "Newcastle", "races": 10}
], (
    "Expected the source-backed GB positive control to contain only "
    f"10 Newcastle races; observed {v4_gb_course_counts.to_dict('records')}"
)

assert len(v4_all_positive_control) == 12, (
    "Expected 12 source-backed races of all jurisdictions on the date; "
    f"observed {len(v4_all_positive_control)}"
)

print("Source-backed Database v4 GB races:", len(v4_gb_positive_control))
display(v4_gb_course_counts)
print()
print("Source-backed Database v4 races by jurisdiction:")
display(v4_jurisdiction_counts)


Source-backed Database v4 GB races: 10


,governed_racecourse_name,races
0,Newcastle,10



Source-backed Database v4 races by jurisdiction:


,candidate_jurisdiction,races
0,Great Britain,10
1,South Africa,2


### Finding — 6 June 2020 is the audit's positive control

The checkpoint evidence entering Notebook 26 establishes the expected contrast:

- official BHA race lists: **28 GB races**;
- source-backed Database v4 GB population: **10 races**;
- missing from the source-backed population: **18 races**;
- missing fixtures: **Newmarket (9)** and **Lingfield Park (9)**;
- Newcastle is present with the correct **10-race** count;
- the general source-backed date population also contains the previously identified **2 South African races**.

The two preceding cells are a fail-loud reproducibility check of that known state. They deliberately avoid the prohibited Source Version 1 off-time field.

This positive control establishes that the screening machinery can detect a known population omission while preserving the audit's source-field prohibition.

## Next bounded question

> **Across 2019, 2020 and 2021, do official BHA race-list counts and the source-backed Inside Rails GB race population agree in the two control years while diverging materially in 2020?**

The next stage should remain a **screening comparison**. It should first establish annual/date/course count patterns. Any mismatch found there remains a candidate discrepancy until reconciled; it must not immediately be labelled a genuine missing race.


In [5]:
# First wider screen: official BHA annual race totals versus the
# source-backed Database v4 GB race population.
#
# BHA source:
# Full Year Racing Data Pack 2021, "Races Ran".
#
# These are screening counts only. Any difference remains a candidate
# discrepancy until reconciled at finer granularity.

bha_annual_races = pd.DataFrame(
    [
        {"year": 2019, "bha_races": 10087},
        {"year": 2020, "bha_races": 7874},
        {"year": 2021, "bha_races": 10354},
    ]
)

with connect_read_only(DATABASE_V4) as conn:
    v4_annual_races = pd.read_sql_query(
        """
        SELECT
            CAST(substr(raw_date, 1, 4) AS INTEGER) AS year,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2019-01-01'
          AND raw_date < '2022-01-01'
        GROUP BY CAST(substr(raw_date, 1, 4) AS INTEGER)
        ORDER BY year
        """,
        conn,
    )

annual_screen = bha_annual_races.merge(
    v4_annual_races,
    on="year",
    how="left",
    validate="one_to_one",
)

annual_screen["v4_minus_bha"] = (
    annual_screen["v4_races"] - annual_screen["bha_races"]
)

annual_screen["missing_vs_bha"] = (
    annual_screen["bha_races"] - annual_screen["v4_races"]
)

display(annual_screen)

,year,bha_races,v4_races,v4_minus_bha,missing_vs_bha
0,2019,10087,10085,-2,2
1,2020,7874,6287,-1587,1587
2,2021,10354,10353,-1,1


### Finding — the population defect is overwhelmingly concentrated in 2020

The first wider screening comparison strongly supports the working hypothesis.

| Year | Official BHA races | Source-backed Database v4 races | Shortfall versus BHA |
|---|---:|---:|---:|
| 2019 | 10,087 | 10,085 | 2 |
| 2020 | 7,874 | 6,287 | 1,587 |
| 2021 | 10,354 | 10,353 | 1 |

The two control years are near-complete at annual level, differing from the later BHA annual totals by only **2 races in 2019** and **1 race in 2021**. Those small differences remain candidate discrepancies and are not yet classified as missing source races.

By contrast, the source-backed 2020 population is **1,587 races below** the official BHA annual total. This is orders of magnitude larger than either adjacent-year control and establishes that the population-completeness problem is overwhelmingly concentrated in 2020.

This does not yet establish that all 1,587 races are genuinely absent from Source Version 1. Annual counts are screening evidence. However, combined with the already reconciled 6 June 2020 positive control, the result justifies localising the 2020 shortfall before expanding the audit to other years.

## Next bounded question

> **How is the 1,587-race 2020 shortfall distributed through the year?**

The next step is to compare 2020 by month before examining individual racecourses or races.

In [6]:
sorted(bha_fixtures[0].keys())

['BSTime',
 'abandonedReasonCode',
 'bcsEvent',
 'blackTypeRaces',
 'courseId',
 'courseName',
 'distance',
 'entriesAvailable',
 'firstRace',
 'fixtureDate',
 'fixtureId',
 'fixtureName',
 'fixtureSession',
 'fixtureType',
 'fixtureYear',
 'going',
 'goingUpdatedDate',
 'goingUpdatedTime',
 'majorEvent',
 'meetingId',
 'numberOfRaces',
 'other',
 'racePlanningCode',
 'racecardAvailable',
 'racingTrackType',
 'rail',
 'resultsAvailable',
 'stalls',
 'ticketsLink',
 'transparentAvailable',
 'weather']

In [7]:
# Validate BHA fixture.numberOfRaces against the authoritative
# fixture race-list records on the known positive-control date.

bha_fixture_declared_counts = pd.DataFrame(
    [
        {
            "course_name": fixture["courseName"],
            "fixture_id": fixture["fixtureId"],
            "numberOfRaces": fixture["numberOfRaces"],
        }
        for fixture in bha_fixtures
    ]
)

bha_count_validation = bha_fixture_declared_counts.merge(
    pd.DataFrame(bha_fixture_race_counts),
    on=["course_name", "fixture_id"],
    how="outer",
    validate="one_to_one",
)

bha_count_validation["difference"] = (
    bha_count_validation["numberOfRaces"]
    - bha_count_validation["races"]
)

display(bha_count_validation)

assert (bha_count_validation["difference"] == 0).all()

,course_name,fixture_id,numberOfRaces,fixture_year,races,difference
0,Chepstow,658,7,2020,0,7
1,Doncaster,718,7,2020,0,7
2,Epsom Downs,766,7,2020,0,7
3,Hexham,931,7,2020,0,7
4,Lingfield Park,19014,9,2020,9,0
5,Musselburgh,12730,7,2020,0,7
6,Newcastle,19006,10,2020,10,0
7,Newmarket,19002,9,2020,9,0
8,Worcester,1833,7,2020,0,7


AssertionError: 

### Finding — `numberOfRaces` is not an observed-race count

The BHA fixture-level `numberOfRaces` field cannot be used directly as the official population of races actually run.

On the 6 June 2020 positive-control date:

- Newcastle declares 10 and has 10 race-list records;
- Newmarket declares 9 and has 9;
- Lingfield Park declares 9 and has 9;
- six other fixture records each declare 7 races but have **zero** race-list records.

Therefore `numberOfRaces` retains programmed/scheduled race counts for fixtures that did not produce races. Using it across 2020 would materially overstate the official race population.

The authoritative population measure remains the BHA fixture race-list records.

In [8]:
bha_abandonment_check = pd.DataFrame(
    [
        {
            "course_name": fixture["courseName"],
            "fixture_id": fixture["fixtureId"],
            "numberOfRaces": fixture["numberOfRaces"],
            "abandonedReasonCode": fixture["abandonedReasonCode"],
        }
        for fixture in bha_fixtures
    ]
).merge(
    pd.DataFrame(bha_fixture_race_counts),
    on=["course_name", "fixture_id"],
    how="outer",
    validate="one_to_one",
)

display(bha_abandonment_check)

,course_name,fixture_id,numberOfRaces,abandonedReasonCode,fixture_year,races
0,Chepstow,658,7,1,2020,0
1,Doncaster,718,7,1,2020,0
2,Epsom Downs,766,7,1,2020,0
3,Hexham,931,7,1,2020,0
4,Lingfield Park,19014,9,0,2020,9
5,Musselburgh,12730,7,1,2020,0
6,Newcastle,19006,10,0,2020,10
7,Newmarket,19002,9,0,2020,9
8,Worcester,1833,7,1,2020,0


### Finding — abandoned fixtures explain the false `numberOfRaces` counts on the positive-control date

On 6 June 2020, every fixture whose BHA race-list endpoint returned zero races has `abandonedReasonCode = 1`.

The three fixtures that produced races — Newcastle, Newmarket and Lingfield Park — all have `abandonedReasonCode = 0`, and their `numberOfRaces` values exactly match their race-list counts.

This suggests that `numberOfRaces` may be usable as a cheap screening count **only after excluding abandoned fixtures**, but the relationship has so far been validated on one date only. It must therefore be tested over a wider period before being trusted for the 2020 monthly localisation.

In [9]:
{k: v for k, v in fixture_payload.items() if k != "data"}

{'current_page': 1,
 'first_page_url': 'https://api09.horseracing.software/bha/v1/fixtures?page=1',
 'from': 1,
 'last_page': 1,
 'last_page_url': 'https://api09.horseracing.software/bha/v1/fixtures?page=1',
 'links': [{'url': None, 'label': '&laquo; Previous', 'active': False},
  {'url': 'https://api09.horseracing.software/bha/v1/fixtures?page=1',
   'label': '1',
   'active': True},
  {'url': None, 'label': 'Next &raquo;', 'active': False}],
 'next_page_url': None,
 'path': 'https://api09.horseracing.software/bha/v1/fixtures',
 'per_page': 100,
 'prev_page_url': None,
 'to': 9,
 'total': 9}

### Finding — the BHA fixture endpoint exposes usable pagination metadata

The BHA fixture response reports standard pagination fields including `current_page`, `last_page`, `per_page`, `next_page_url` and `total`.

The positive-control query returned nine fixture records on one page. Wider date-range screening can therefore iterate pages explicitly rather than assuming a single response contains the complete fixture population.

In [10]:
from calendar import monthrange


def get_bha_fixtures(from_date: str, to_date: str) -> list[dict]:
    rows = []
    page = 1

    while True:
        params = {
            "fromdate": from_date,
            "todate": to_date,
            "resultsAvailable": 1,
            "order": "asc",
            "page": page,
            "per_page": 100,
        }

        url = (
            f"{BHA_API_ROOT}/bha/v1/fixtures/?"
            + urlencode(params)
        )

        payload = get_bha_json(url)
        rows.extend(payload["data"])

        if page >= payload["last_page"]:
            break

        page += 1
        time.sleep(1.0)

    return rows


monthly_rows = []

for month in range(1, 13):
    last_day = monthrange(2020, month)[1]

    from_date = f"2020-{month:02d}-01"
    to_date = f"2020-{month:02d}-{last_day:02d}"

    fixtures = get_bha_fixtures(from_date, to_date)

    fixture_df = pd.DataFrame(fixtures)

    fixture_df["abandonedReasonCode"] = pd.to_numeric(
        fixture_df["abandonedReasonCode"],
        errors="coerce",
    )

    fixture_df["numberOfRaces"] = pd.to_numeric(
        fixture_df["numberOfRaces"],
        errors="coerce",
    )

    active = fixture_df[
        fixture_df["abandonedReasonCode"].fillna(0) == 0
    ]

    abandoned = fixture_df[
        fixture_df["abandonedReasonCode"].fillna(0) != 0
    ]

    monthly_rows.append(
        {
            "month": f"2020-{month:02d}",
            "fixture_records": len(fixture_df),
            "active_fixtures": len(active),
            "abandoned_fixtures": len(abandoned),
            "screened_races": int(active["numberOfRaces"].sum()),
        }
    )

    time.sleep(1.0)


bha_2020_monthly_screen = pd.DataFrame(monthly_rows)

display(bha_2020_monthly_screen)

print(
    "2020 screened BHA races:",
    bha_2020_monthly_screen["screened_races"].sum(),
)

print("Official BHA 2020 races:", 7874)

print(
    "Difference:",
    bha_2020_monthly_screen["screened_races"].sum() - 7874,
)

,month,fixture_records,active_fixtures,abandoned_fixtures,screened_races
0,2020-01,112,104,8,736
1,2020-02,113,88,25,601
2,2020-03,109,55,54,368
3,2020-04,129,0,129,0
4,2020-05,223,0,223,0
5,2020-06,244,92,152,809
6,2020-07,181,106,75,915
7,2020-08,193,100,93,872
8,2020-09,146,123,23,953
9,2020-10,136,127,9,1004


2020 screened BHA races: 7889
Official BHA 2020 races: 7874
Difference: 15


### Finding — non-abandoned `numberOfRaces` is a useful but imperfect screening measure

Across 2020, excluding fixtures with a non-zero `abandonedReasonCode` and summing the remaining fixture-level `numberOfRaces` produces **7,889 races**.

The official BHA full-year total is **7,874 races**, leaving a residual overcount of **15 races**.

Therefore the combination of `abandonedReasonCode = 0` and `numberOfRaces` is not an exact measure of races actually run. Some additional within-fixture or race-level changes must remain.

However, the residual error is small relative to the **1,587-race annual source-backed shortfall** already identified. The fixture-derived monthly totals can therefore be used as a cheap **screening measure to localise the 2020 problem by month**, but they must not be treated as final official monthly race counts. Candidate mismatches will subsequently be verified using BHA race-list records.

In [11]:
# Localise the 2020 source-population shortfall by month.
#
# BHA screened_races is deliberately treated as approximate screening evidence:
# its annual total is 15 above the official BHA race total.
# Database v4 remains the exact source-backed comparator.

with connect_read_only(DATABASE_V4) as conn:
    v4_2020_monthly = pd.read_sql_query(
        """
        SELECT
            substr(raw_date, 1, 7) AS month,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-01-01'
          AND raw_date < '2021-01-01'
        GROUP BY substr(raw_date, 1, 7)
        ORDER BY month
        """,
        conn,
    )

monthly_population_screen = bha_2020_monthly_screen[
    ["month", "screened_races"]
].merge(
    v4_2020_monthly,
    on="month",
    how="left",
    validate="one_to_one",
)

monthly_population_screen["v4_races"] = (
    monthly_population_screen["v4_races"]
    .fillna(0)
    .astype(int)
)

monthly_population_screen["screened_shortfall"] = (
    monthly_population_screen["screened_races"]
    - monthly_population_screen["v4_races"]
)

display(monthly_population_screen)

print(
    "Screened annual shortfall:",
    monthly_population_screen["screened_shortfall"].sum(),
)

print(
    "Known exact annual shortfall:",
    7874 - 6287,
)

,month,screened_races,v4_races,screened_shortfall
0,2020-01,736,584,152
1,2020-02,601,473,128
2,2020-03,368,243,125
3,2020-04,0,0,0
4,2020-05,0,0,0
5,2020-06,809,260,549
6,2020-07,915,275,640
7,2020-08,872,865,7
8,2020-09,953,953,0
9,2020-10,1004,1002,2


Screened annual shortfall: 1602
Known exact annual shortfall: 1587


In [12]:
# Exact clean-month control: September 2020.
#
# Fetch every BHA fixture record for the month, then count the actual
# race-list records returned for each fixture. This is the authoritative
# population measure used by this audit.

SEPTEMBER_FROM = "2020-09-01"
SEPTEMBER_TO = "2020-09-30"

september_fixtures = get_bha_fixtures(
    SEPTEMBER_FROM,
    SEPTEMBER_TO,
)

september_exact_rows = []

for fixture in september_fixtures:
    race_list_url = (
        f"{BHA_API_ROOT}/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/"
        f"{fixture['fixtureId']}/races"
    )

    race_payload = get_bha_json(race_list_url)
    race_rows = race_payload["data"]

    september_exact_rows.append(
        {
            "course_name": fixture["courseName"],
            "fixture_id": fixture["fixtureId"],
            "abandonedReasonCode": fixture["abandonedReasonCode"],
            "numberOfRaces": fixture["numberOfRaces"],
            "actual_races": len(race_rows),
        }
    )

    time.sleep(1.5)


september_exact = pd.DataFrame(september_exact_rows)

bha_september_exact_total = int(
    september_exact["actual_races"].sum()
)

with connect_read_only(DATABASE_V4) as conn:
    v4_september_total = pd.read_sql_query(
        """
        SELECT COUNT(*) AS races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-09-01'
          AND raw_date < '2020-10-01'
        """,
        conn,
    ).iloc[0]["races"]

v4_september_total = int(v4_september_total)

print("BHA fixture records:", len(september_fixtures))
print("Exact BHA race-list records:", bha_september_exact_total)
print("Database v4 races:", v4_september_total)
print(
    "BHA minus v4:",
    bha_september_exact_total - v4_september_total,
)

print()
print("Fixture-level programmed-versus-actual differences:")

september_fixture_differences = september_exact[
    pd.to_numeric(
        september_exact["numberOfRaces"],
        errors="coerce",
    )
    != september_exact["actual_races"]
].copy()

display(september_fixture_differences)

BHA fixture records: 146
Exact BHA race-list records: 954
Database v4 races: 953
BHA minus v4: 1

Fixture-level programmed-versus-actual differences:


,course_name,fixture_id,abandonedReasonCode,numberOfRaces,actual_races
78,Southwell,917,1,7,0
132,Epsom Downs,1430,1,1,0
139,Worcester,1558,1,7,0


In [13]:
# Persistent BHA API cache.
#
# Every successful BHA API JSON response is preserved as the exact response
# bytes plus a metadata sidecar. Authorization credentials are never written.
#
# Cache is under data/cache/, which is already excluded from Git.

from datetime import datetime, timezone
import os


BHA_CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "cache"
    / "bha"
    / "api_json"
)

BHA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

BHA_CACHE_STATS = {
    "hits": 0,
    "misses": 0,
}


def bha_cache_paths(url: str) -> tuple[Path, Path]:
    cache_key = hashlib.sha256(
        url.encode("utf-8")
    ).hexdigest()

    body_path = BHA_CACHE_DIR / f"{cache_key}.json"
    metadata_path = BHA_CACHE_DIR / f"{cache_key}.meta.json"

    return body_path, metadata_path


def get_bha_json(
    url: str,
    *,
    refresh: bool = False,
) -> dict:
    body_path, metadata_path = bha_cache_paths(url)

    # Default behaviour is cache-first.
    if body_path.is_file() and not refresh:
        raw_body = body_path.read_bytes()

        assert metadata_path.is_file(), (
            f"BHA cache body exists without metadata: {body_path}"
        )

        metadata = json.loads(
            metadata_path.read_text(encoding="utf-8")
        )

        assert metadata["requested_url"] == url, (
            "BHA cache URL mismatch"
        )

        observed_sha256 = hashlib.sha256(
            raw_body
        ).hexdigest()

        assert observed_sha256 == metadata["body_sha256"], (
            f"BHA cache SHA-256 mismatch: {body_path}"
        )

        BHA_CACHE_STATS["hits"] += 1

        return json.loads(
            raw_body.decode("utf-8")
        )

    request = Request(
        url,
        headers={
            "Authorization": authorization_value,
            "Accept": "application/json",
            "Origin": "https://www.britishhorseracing.com",
            "Referer": "https://www.britishhorseracing.com/",
            "User-Agent": USER_AGENT,
        },
    )

    with urlopen(request, timeout=30) as response:
        raw_body = response.read()

        response_status = getattr(
            response,
            "status",
            None,
        )

        content_type = response.headers.get(
            "Content-Type"
        )

        resolved_url = response.geturl()

    # Fail before caching if the response is not valid JSON.
    payload = json.loads(
        raw_body.decode("utf-8")
    )

    body_sha256 = hashlib.sha256(
        raw_body
    ).hexdigest()

    metadata = {
        "requested_url": url,
        "resolved_url": resolved_url,
        "fetched_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "http_status": response_status,
        "content_type": content_type,
        "body_bytes": len(raw_body),
        "body_sha256": body_sha256,
    }

    # Atomic writes prevent an interrupted request from leaving
    # an apparently valid partial cache entry.
    body_tmp = Path(str(body_path) + ".tmp")
    metadata_tmp = Path(str(metadata_path) + ".tmp")

    body_tmp.write_bytes(raw_body)

    metadata_tmp.write_text(
        json.dumps(
            metadata,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(body_tmp, body_path)
    os.replace(metadata_tmp, metadata_path)

    BHA_CACHE_STATS["misses"] += 1

    return payload


print("BHA API cache:", BHA_CACHE_DIR)
print("Cache-first BHA acquisition helper ready.")

BHA API cache: /home/rob/Documents/inside-rails-horse-racing/data/cache/bha/api_json
Cache-first BHA acquisition helper ready.


In [14]:
print(BHA_CACHE_DIR)
print("Existing cached responses:", len(list(BHA_CACHE_DIR.glob("*.json"))))
print(BHA_CACHE_STATS)

/home/rob/Documents/inside-rails-horse-racing/data/cache/bha/api_json
Existing cached responses: 0
{'hits': 0, 'misses': 0}


### Finding — September 2020 is near-complete but not yet a clean control month

An exact BHA race-list count for September 2020 returned **954 races**, compared with **953 source-backed races in Database v4**.

September therefore contains a **one-race candidate population discrepancy** and cannot yet be classified as a completely clean control month.

This exact result also differs from the earlier fixture-level screening count of 953 BHA races. In the exact pass, the only fixture-level differences between `numberOfRaces` and race-list records were three wholly abandoned fixtures. The reason for the 953-versus-954 difference between the two BHA retrieval passes is therefore unresolved.

Because the earlier BHA responses were not persisted, that difference cannot be reconstructed reliably from those requests. All subsequent BHA API responses will therefore be cached as immutable local evidence before further reconciliation.

In [16]:
print("Cache stats:", BHA_CACHE_STATS)
print(
    "Cached response bodies:",
    len(list(BHA_CACHE_DIR.glob("*.json"))) // 2,
)

Cache stats: {'hits': 0, 'misses': 0}
Cached response bodies: 0


In [17]:
# Localise the exact September 2020 discrepancy by date.
# No further BHA requests are made here.

september_fixture_dates = pd.DataFrame(
    [
        {
            "fixture_id": fixture["fixtureId"],
            "fixture_date": pd.to_datetime(
                fixture["fixtureDate"]
            ).strftime("%Y-%m-%d"),
        }
        for fixture in september_fixtures
    ]
)

bha_september_daily = (
    september_exact
    .merge(
        september_fixture_dates,
        on="fixture_id",
        how="left",
        validate="one_to_one",
    )
    .groupby("fixture_date", as_index=False)["actual_races"]
    .sum()
    .rename(columns={"actual_races": "bha_races"})
)

with connect_read_only(DATABASE_V4) as conn:
    v4_september_daily = pd.read_sql_query(
        """
        SELECT
            raw_date AS fixture_date,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-09-01'
          AND raw_date < '2020-10-01'
        GROUP BY raw_date
        ORDER BY raw_date
        """,
        conn,
    )

september_daily_check = bha_september_daily.merge(
    v4_september_daily,
    on="fixture_date",
    how="outer",
)

september_daily_check[["bha_races", "v4_races"]] = (
    september_daily_check[["bha_races", "v4_races"]]
    .fillna(0)
    .astype(int)
)

september_daily_check["difference"] = (
    september_daily_check["bha_races"]
    - september_daily_check["v4_races"]
)

display(
    september_daily_check[
        september_daily_check["difference"] != 0
    ]
)

,fixture_date,bha_races,v4_races,difference
24,2020-09-25,32,31,1


### Finding — September's one-race discrepancy is confined to 25 September 2020

The exact September comparison localises the entire monthly discrepancy to a single date.

On **25 September 2020**, the official BHA race lists contain **32 races**, while Database v4 contains **31 source-backed GB races**.

All other September dates reconcile at daily population-count level.

The remaining September question is therefore bounded to identifying which racecourse/fixture accounts for the single candidate missing race on 25 September.

In [19]:
# Localise the 25 September 2020 discrepancy by racecourse/fixture.

september_exact_dated = september_exact.merge(
    september_fixture_dates,
    on="fixture_id",
    how="left",
    validate="one_to_one",
)

bha_sep25 = (
    september_exact_dated[
        september_exact_dated["fixture_date"] == "2020-09-25"
    ][
        [
            "course_name",
            "fixture_id",
            "actual_races",
        ]
    ]
    .sort_values("course_name")
)

print("BHA — 25 September 2020")
display(bha_sep25)


with connect_read_only(DATABASE_V4) as conn:
    v4_sep25 = pd.read_sql_query(
        """
        SELECT
            governed_racecourse_name,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date = '2020-09-25'
        GROUP BY governed_racecourse_name
        ORDER BY governed_racecourse_name
        """,
        conn,
    )

print("Database v4 — 25 September 2020")
display(v4_sep25)


BHA — 25 September 2020


,course_name,fixture_id,actual_races
120,Haydock Park,889,7
121,Newcastle,12398,9
119,Newmarket,1242,7
118,Uttoxeter,19426,8
122,Worcester,1838,1


Database v4 — 25 September 2020


,governed_racecourse_name,v4_races
0,Haydock Park,7
1,Newcastle,9
2,Newmarket — Rowley Mile,7
3,Uttoxeter,8


### Finding — September's candidate missing race is at Worcester on 25 September 2020

The one-race September discrepancy has been localised completely.

On **25 September 2020**, the BHA race-list data contains:

| Racecourse | BHA races | Database v4 races |
|---|---:|---:|
| Haydock Park | 7 | 7 |
| Newcastle | 9 | 9 |
| Newmarket — Rowley Mile | 7 | 7 |
| Uttoxeter | 8 | 8 |
| Worcester | **1** | **0** |

The four other racecourses reconcile exactly.

The September discrepancy is therefore not caused by the fixture-level screening approximation. It is a single official BHA race at **Worcester** for which Database v4 contains no corresponding source-backed Great Britain race occurrence.

The next question is to identify that Worcester race from the BHA race-list record and confirm whether Source Version 1 itself contains any Worcester source records for that date.

In [20]:
# Inspect the single official Worcester race on 25 September 2020.

worcester_sep25_url = (
    f"{BHA_API_ROOT}/bha/v1/fixtures/2020/1838/races"
)

worcester_sep25_payload = get_bha_json(
    worcester_sep25_url
)

worcester_sep25_races = worcester_sep25_payload["data"]

print(
    "BHA Worcester races:",
    len(worcester_sep25_races),
)

display(
    pd.DataFrame(worcester_sep25_races)
)

BHA Worcester races: 1


,raceId,yearOfRace,divisionSequence,raceDate,raceTime,raceName,ageLimit,prizeAmount,prizeCurrency,raceClass,...,goingText,currentStageCode,transparentWindowStatus,raceCriteriaRaceType,abandonedReasonCode,blackTypeRace,distanceChangeText,plus10,winnersDetails,aroRace
0,8816,2020,0,2020-09-25,21:01:00,THE CHALKDUST RACING CLUB SELLING HURDLE RACE ...,4-7YO,3900,GBP,5,...,None,0,None,JUMP,0,0,2m 7f,False,[],False


In [21]:
# Inspect every field on the anomalous Worcester BHA race-list record.

worcester_sep25_record = pd.Series(
    worcester_sep25_races[0]
)

display(
    worcester_sep25_record
    .rename("value")
    .to_frame()
)

,value
raceId,8816
yearOfRace,2020
divisionSequence,0
raceDate,2020-09-25
raceTime,21:01:00
raceName,THE CHALKDUST RACING CLUB SELLING HURDLE RACE ...
ageLimit,4-7YO
prizeAmount,3900
prizeCurrency,GBP
raceClass,5


In [22]:
# Inspect the BHA fixture-detail record for Worcester on 25 September 2020.
#
# This is a separate official BHA representation from the race-list endpoint
# and may expose whether the fixture/race ever progressed to a result state.

worcester_sep25_fixture_url = (
    f"{BHA_API_ROOT}/bha/v1/fixtures/2020/1838"
)

worcester_sep25_fixture = get_bha_json(
    worcester_sep25_fixture_url
)

display(
    pd.Series(worcester_sep25_fixture)
    .rename("value")
    .to_frame()
)

,value
success,True
total,1
data,"[{'fixtureId': 1838, 'fixtureYear': '2020', 'f..."


In [23]:
# Expand the Worcester fixture-detail record.

worcester_fixture_record = worcester_sep25_fixture["data"][0]

display(
    pd.Series(worcester_fixture_record)
    .rename("value")
    .to_frame()
)

,value
fixtureId,1838
fixtureYear,2020
fixtureDate,2020-09-25
meetingId,1423
courseId,59
courseName,Worcester
ticketsLink,http://www.worcester-racecourse.co.uk/horse-ra...
BSTime,01:29:19
abandonedReasonCode,1
fixtureType,JUMP


### Finding — September 2020 reconciles exactly once abandoned-fixture race-list ghosts are excluded

The initial BHA race-list enumeration returned 954 records for September 2020, compared with 953 source-backed Database v4 races.

The apparent extra BHA race was localised to Worcester on 25 September 2020. The BHA race-list endpoint retains one programmed Worcester race for that fixture, but the BHA fixture-detail record establishes that the entire fixture was **abandoned 72 hours before racing**:

- `abandonedReasonCode = 1`;
- `resultsAvailable = 0`;
- `goingText = "ABANDONED - Abandoned (72 Hours Before)"`.

The Worcester race-list record therefore does **not** represent a race that ran.

Once race-list records belonging to wholly abandoned fixtures are excluded, the September 2020 BHA population is **953 races**, exactly matching the **953 source-backed Database v4 races**.

September is therefore established as a clean control month at race-population level.

### Methodological consequence

The BHA race-list endpoint cannot be treated as an unconditional list of races that ran. It may retain programmed race records for fixtures that were subsequently abandoned.

For exact population reconciliation, fixture abandonment status must therefore be applied before counting race-list records.

In [25]:
# Confirm September 2020 population reconciliation after excluding
# race-list records attached to wholly abandoned fixtures.

september_exact["abandoned_num"] = (
    pd.to_numeric(
        september_exact["abandonedReasonCode"],
        errors="coerce",
    )
    .fillna(0)
    .astype(int)
)

september_completed = september_exact[
    september_exact["abandoned_num"] == 0
]

september_abandoned_with_race_rows = september_exact[
    (september_exact["abandoned_num"] != 0)
    & (september_exact["actual_races"] > 0)
]

bha_completed_races = int(
    september_completed["actual_races"].sum()
)

with connect_read_only(DATABASE_V4) as conn:
    v4_september_races = conn.execute(
        """
        SELECT COUNT(*)
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-09-01'
          AND raw_date < '2020-10-01'
        """
    ).fetchone()[0]

print(
    "BHA completed-fixture race-list records:",
    bha_completed_races,
)

print(
    "Database v4 races:",
    v4_september_races,
)

print(
    "Difference:",
    bha_completed_races - v4_september_races,
)

print(
    "\nRace-list records retained under abandoned fixtures:"
)

display(
    september_abandoned_with_race_rows[
        [
            "course_name",
            "fixture_id",
            "abandonedReasonCode",
            "actual_races",
        ]
    ]
)

BHA completed-fixture race-list records: 953
Database v4 races: 953
Difference: 0

Race-list records retained under abandoned fixtures:


,course_name,fixture_id,abandonedReasonCode,actual_races
122,Worcester,1838,1,1


### Finding — September 2020 reconciles exactly after excluding wholly abandoned fixtures

The apparent one-race September discrepancy has been resolved.

The BHA race-list endpoint initially returned **954 race records**, while Database v4 contained **953 source-backed Great Britain races**.

The extra BHA record belonged to Worcester fixture `1838` on 25 September 2020. BHA fixture-detail evidence establishes that this fixture was wholly abandoned:

- `abandonedReasonCode = 1`;
- `resultsAvailable = 0`;
- `goingText = "ABANDONED - Abandoned (72 Hours Before)"`.

Despite that abandonment, the fixture's race-list endpoint retains one programmed race record.

After excluding race-list records attached to wholly abandoned fixtures:

- BHA completed-fixture race-list records: **953**
- Database v4 races: **953**
- Difference: **0**

September 2020 is therefore a clean control month at race-population level.

### Methodological consequence

Presence in the BHA race-list endpoint is not sufficient evidence that a race ran. Programmed race records may survive for wholly abandoned fixtures.

Exact BHA population reconciliation must therefore apply fixture-level abandonment status before counting race-list records.

In [27]:
# Exact June 2020 BHA race-population count.
#
# Method:
# 1. retrieve every June fixture;
# 2. exclude fixtures the BHA marks as wholly abandoned;
# 3. count actual race-list records only for the remaining fixtures;
# 4. compare with the source-backed Database v4 population.

june_fixture_url = (
    f"{BHA_API_ROOT}/bha/v1/fixtures/"
    "?fromdate=2020-06-01"
    "&todate=2020-06-30"
    "&resultsAvailable=1"
    "&order=asc"
    "&page=1"
    "&per_page=100"
)

first_page = get_bha_json(june_fixture_url)

june_fixtures = list(first_page["data"])

for page in range(2, int(first_page["last_page"]) + 1):
    page_url = (
        f"{BHA_API_ROOT}/bha/v1/fixtures/"
        "?fromdate=2020-06-01"
        "&todate=2020-06-30"
        "&resultsAvailable=1"
        "&order=asc"
        f"&page={page}"
        "&per_page=100"
    )
    page_payload = get_bha_json(page_url)
    june_fixtures.extend(page_payload["data"])


june_active_fixtures = [
    fixture
    for fixture in june_fixtures
    if int(fixture.get("abandonedReasonCode") or 0) == 0
]

june_abandoned_fixtures = [
    fixture
    for fixture in june_fixtures
    if int(fixture.get("abandonedReasonCode") or 0) != 0
]


june_exact_rows = []

for fixture in june_active_fixtures:
    race_url = (
        f"{BHA_API_ROOT}/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/{fixture['fixtureId']}/races"
    )

    race_payload = get_bha_json(race_url)
    races = race_payload["data"]

    june_exact_rows.append(
        {
            "fixture_date": pd.to_datetime(
                fixture["fixtureDate"]
            ).strftime("%Y-%m-%d"),
            "course_name": fixture["courseName"],
            "fixture_id": fixture["fixtureId"],
            "fixture_year": fixture["fixtureYear"],
            "actual_races": len(races),
        }
    )

    time.sleep(1.5)


june_exact = pd.DataFrame(june_exact_rows)

bha_june_races = int(
    june_exact["actual_races"].sum()
)

with connect_read_only(DATABASE_V4) as conn:
    v4_june_races = conn.execute(
        """
        SELECT COUNT(*)
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-06-01'
          AND raw_date < '2020-07-01'
        """
    ).fetchone()[0]


print("BHA fixture records:", len(june_fixtures))
print("BHA active fixtures:", len(june_active_fixtures))
print("BHA wholly abandoned fixtures:", len(june_abandoned_fixtures))
print("BHA completed-fixture race-list records:", bha_june_races)
print("Database v4 races:", v4_june_races)
print("Exact June shortfall:", bha_june_races - v4_june_races)

BHA fixture records: 244
BHA active fixtures: 92
BHA wholly abandoned fixtures: 152
BHA completed-fixture race-list records: 809
Database v4 races: 260
Exact June shortfall: 549


### Finding — June 2020 has a 549-race source-backed population shortfall

The corrected BHA reconciliation method was applied across the whole of June 2020.

The BHA fixture population contains:

- 244 fixture records;
- 92 non-abandoned fixtures;
- 152 wholly abandoned fixtures.

After excluding wholly abandoned fixtures, the BHA race-list endpoints contain **809 race records** attached to the 92 active fixtures.

Database v4 contains **260 source-backed Great Britain race occurrences** for the same month.

The resulting June population shortfall is therefore:

**809 - 260 = 549 races**

This exactly matches the earlier fixture-level screening shortfall of 549 races.

The June deficit is therefore not caused by wholly abandoned fixtures or by the earlier fixture-level counting approximation. It represents a material difference between the BHA active-fixture race population and Source Version 1 / Database v4.

The next step is to localise the 549-race June deficit by date before investigating individual racecourses or races.

In [28]:
# Localise the June 2020 population deficit by date.

bha_june_by_date = (
    june_exact
    .groupby("fixture_date", as_index=False)["actual_races"]
    .sum()
    .rename(columns={"actual_races": "bha_races"})
)

with connect_read_only(DATABASE_V4) as conn:
    v4_june_by_date = pd.read_sql_query(
        """
        SELECT
            raw_date AS fixture_date,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-06-01'
          AND raw_date < '2020-07-01'
        GROUP BY raw_date
        ORDER BY raw_date
        """,
        conn,
    )

june_by_date = (
    bha_june_by_date
    .merge(
        v4_june_by_date,
        on="fixture_date",
        how="outer",
    )
    .fillna(0)
)

june_by_date["bha_races"] = june_by_date["bha_races"].astype(int)
june_by_date["v4_races"] = june_by_date["v4_races"].astype(int)

june_by_date["difference"] = (
    june_by_date["bha_races"]
    - june_by_date["v4_races"]
)

june_discrepant_dates = june_by_date[
    june_by_date["difference"] != 0
].sort_values("fixture_date")

display(june_discrepant_dates)

print(
    "Number of discrepant dates:",
    len(june_discrepant_dates),
)

print(
    "Total June difference:",
    int(june_discrepant_dates["difference"].sum()),
)

,fixture_date,bha_races,v4_races,difference
3,2020-06-04,20,10,10
4,2020-06-05,19,0,19
5,2020-06-06,28,10,18
6,2020-06-07,29,0,29
7,2020-06-08,29,10,19
8,2020-06-09,28,9,19
9,2020-06-10,35,17,18
10,2020-06-11,32,8,24
11,2020-06-12,27,9,18
12,2020-06-13,35,9,26


Number of discrepant dates: 27
Total June difference: 549


### Finding — the June 2020 defect begins on 4 June and persists through month-end

The exact June population shortfall was localised by race date.

Database v4 reconciles with the BHA population for the first three days of June 2020. From **4 June through 30 June**, however, every date is deficient.

There are:

- **27 discrepant dates**;
- **549 races missing relative to the BHA completed-fixture population**;
- no intervening clean date after the defect begins.

Several dates contain no Database v4 races despite official BHA racing taking place, including 5, 7, 18, 19, 22, 24, 25 and 28 June.

The pattern is therefore not consistent with a small number of isolated missing races or meetings. It indicates a sustained source-population failure beginning on **4 June 2020** and continuing through the end of the month.

The next step is to localise the June shortfall by racecourse to determine whether particular meetings/racecourses are systematically retained while others disappear.

In [29]:
# Localise the exact June 2020 deficit by racecourse.
# Keep BHA and v4 aggregates separate initially so we make no
# undocumented assumptions about name equivalence.

bha_june_by_course = (
    june_exact
    .groupby("course_name", as_index=False)
    .agg(
        bha_fixtures=("fixture_id", "nunique"),
        bha_races=("actual_races", "sum"),
    )
    .sort_values(
        ["bha_races", "course_name"],
        ascending=[False, True],
    )
)

with connect_read_only(DATABASE_V4) as conn:
    v4_june_by_course = pd.read_sql_query(
        """
        SELECT
            governed_racecourse_name,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-06-01'
          AND raw_date < '2020-07-01'
        GROUP BY governed_racecourse_name
        ORDER BY v4_races DESC, governed_racecourse_name
        """,
        conn,
    )

print("BHA completed-fixture population — June 2020")
display(bha_june_by_course)

print("\nDatabase v4 population — June 2020")
display(v4_june_by_course)

BHA completed-fixture population — June 2020


,course_name,bha_fixtures,bha_races
17,Newmarket,11,95
13,Lingfield Park,8,69
11,Kempton Park,8,68
10,Haydock Park,5,50
16,Newcastle,5,49
23,Windsor,5,42
6,Doncaster,4,38
4,Chelmsford City,4,37
0,Ascot,5,36
24,Wolverhampton,4,35



Database v4 population — June 2020


,governed_racecourse_name,v4_races
0,Kempton Park,68
1,Newcastle,49
2,Doncaster,38
3,Chelmsford City,37
4,Great Yarmouth,34
5,Chepstow,26
6,Musselburgh,8


### Finding — June 2020 deficit is concentrated in entire racecourse populations

The June racecourse-level comparison reveals a highly structured defect.

The BHA completed-fixture population contains **809 races across 25 racecourses**.

Database v4 contains **260 races across only seven of those racecourses**:

- Kempton Park — BHA 68 / v4 68
- Newcastle — BHA 49 / v4 49
- Doncaster — BHA 38 / v4 38
- Chelmsford City — BHA 37 / v4 37
- Great Yarmouth — BHA 34 / v4 34
- Chepstow — BHA 26 / v4 26
- Musselburgh — BHA 8 / v4 8

The monthly population at each of these seven racecourses reconciles exactly.

The remaining **18 BHA racecourses are completely absent from Database v4 for June 2020**. Their combined BHA population is **549 races**, exactly equal to the full June shortfall.

This is therefore not presently behaving like scattered individual race omissions. At monthly racecourse level, Source Version 1 appears to have retained complete populations for a subset of racecourses while omitting the entire June population of the others.

The next step is to test the seven retained racecourses at fixture/date level. This will establish whether their apparently complete monthly totals represent genuinely complete individual fixtures rather than compensating discrepancies within a racecourse.

In [30]:
# Test whether every June fixture at the seven retained racecourses
# reconciles individually between BHA and Database v4.

retained_june_courses = set(
    v4_june_by_course["governed_racecourse_name"]
)

bha_retained_by_fixture = (
    june_exact[
        june_exact["course_name"].isin(retained_june_courses)
    ]
    [
        [
            "fixture_date",
            "course_name",
            "fixture_id",
            "actual_races",
        ]
    ]
    .rename(columns={"actual_races": "bha_races"})
)

with connect_read_only(DATABASE_V4) as conn:
    v4_june_by_date_course = pd.read_sql_query(
        """
        SELECT
            raw_date AS fixture_date,
            governed_racecourse_name AS course_name,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-06-01'
          AND raw_date < '2020-07-01'
        GROUP BY
            raw_date,
            governed_racecourse_name
        ORDER BY
            raw_date,
            governed_racecourse_name
        """,
        conn,
    )

v4_retained_by_date_course = (
    v4_june_by_date_course[
        v4_june_by_date_course["course_name"].isin(
            retained_june_courses
        )
    ]
)

retained_fixture_check = (
    bha_retained_by_fixture
    .merge(
        v4_retained_by_date_course,
        on=["fixture_date", "course_name"],
        how="outer",
    )
)

retained_fixture_check["bha_races"] = (
    retained_fixture_check["bha_races"]
    .fillna(0)
    .astype(int)
)

retained_fixture_check["v4_races"] = (
    retained_fixture_check["v4_races"]
    .fillna(0)
    .astype(int)
)

retained_fixture_check["difference"] = (
    retained_fixture_check["bha_races"]
    - retained_fixture_check["v4_races"]
)

retained_fixture_mismatches = retained_fixture_check[
    retained_fixture_check["difference"] != 0
]

print(
    "Retained BHA fixtures:",
    len(bha_retained_by_fixture),
)

print(
    "Retained BHA races:",
    int(bha_retained_by_fixture["bha_races"].sum()),
)

print(
    "Fixture/date-course mismatches:",
    len(retained_fixture_mismatches),
)

display(retained_fixture_mismatches)

print("\nCompletely absent June racecourses:")

bha_absent_june_courses = bha_june_by_course[
    ~bha_june_by_course["course_name"].isin(
        retained_june_courses
    )
]

display(bha_absent_june_courses)

print(
    "Absent racecourses:",
    len(bha_absent_june_courses),
)

print(
    "Absent BHA races:",
    int(bha_absent_june_courses["bha_races"].sum()),
)

Retained BHA fixtures: 29
Retained BHA races: 260
Fixture/date-course mismatches: 0


,fixture_date,course_name,fixture_id,bha_races,v4_races,difference



Completely absent June racecourses:


,course_name,bha_fixtures,bha_races
17,Newmarket,11,95
13,Lingfield Park,8,69
10,Haydock Park,5,50
23,Windsor,5,42
0,Ascot,5,36
24,Wolverhampton,4,35
22,Thirsk,3,30
19,Redcar,3,29
3,Beverley,3,27
12,Leicester,3,27


Absent racecourses: 18
Absent BHA races: 549


### Finding — retained June racecourses are complete fixture by fixture

The seven racecourses represented in Database v4 were tested at individual fixture/date level against the BHA completed-fixture race population.

Results:

- BHA retained fixtures: **29**
- BHA races across those fixtures: **260**
- Database v4 races across those fixtures: **260**
- fixture/date-course mismatches: **0**

Every retained fixture therefore reconciles exactly.

Conversely, the remaining **18 racecourses are completely absent from Database v4 for June 2020**:

- 61 BHA fixtures;
- 549 BHA races;
- 0 Database v4 races.

Those 549 races exactly equal the full June 2020 population shortfall.

The June defect is therefore structured at whole-fixture/racecourse-population level rather than as partial damage within retained meetings. Source Version 1 / Database v4 contains complete retained fixtures and entirely omits the other June racecourse populations.

Before investigating why this particular subset was retained, the next step is to verify directly from the immutable Source Version 1 evidence that the omitted racecourse populations were absent from the source itself rather than lost during later Database v4 reconciliation.

In [31]:
# Inspect which governed GB racecourses have any admitted
# Source Version 1 runner records in June 2020.
#
# This deliberately does NOT use the prohibited Source V1 `off` field.
# We are testing source presence, not reconstructing race identity.

with connect_read_only(DATABASE_V4) as conn:
    source_v1_june_course_presence = pd.read_sql_query(
        """
        SELECT
            ref.racecourse_name AS governed_racecourse_name,
            ref.candidate_course_label AS source_course_label,
            COUNT(raw.source_record_id) AS source_runner_rows,
            COUNT(DISTINCT raw.date) AS source_dates
        FROM view_gb_racecourse_identity_reference AS ref
        JOIN source_raceform_v1_record AS raw
          ON raw.course = ref.candidate_course_label
         AND raw.structural_status = 'admitted_runner_record'
         AND raw.date >= '2020-06-01'
         AND raw.date < '2020-07-01'
        GROUP BY
            ref.racecourse_name,
            ref.candidate_course_label
        ORDER BY
            ref.racecourse_name,
            ref.candidate_course_label
        """,
        conn,
    )

display(source_v1_june_course_presence)

print(
    "Governed GB racecourses represented in raw Source V1:",
    source_v1_june_course_presence[
        "governed_racecourse_name"
    ].nunique(),
)

print(
    "GB Source V1 runner rows in June:",
    int(source_v1_june_course_presence["source_runner_rows"].sum()),
)

,governed_racecourse_name,source_course_label,source_runner_rows,source_dates
0,Chelmsford City,Chelmsford (AW),340,4
1,Chepstow,Chepstow,250,3
2,Doncaster,Doncaster,349,4
3,Great Yarmouth,Yarmouth,290,4
4,Kempton Park,Kempton (AW),683,8
5,Musselburgh,Musselburgh,55,1
6,Newcastle,Newcastle (AW),535,5


Governed GB racecourses represented in raw Source V1: 7
GB Source V1 runner rows in June: 2502


### Finding — the June 2020 population defect originates in Source Version 1

The immutable Source Version 1 raw evidence was inspected directly without using the prohibited `off` field.

For June 2020, admitted Source Version 1 runner records exist for exactly seven governed British racecourses:

- Chelmsford City
- Chepstow
- Doncaster
- Great Yarmouth
- Kempton Park
- Musselburgh
- Newcastle

These are exactly the same seven racecourses represented in Database v4.

Source Version 1 contains **2,502 admitted runner rows across those seven racecourses**, and contains no June runner records for the other 18 BHA racecourses.

Earlier reconciliation established that:

- the seven represented racecourses contain 29 BHA fixtures and 260 races;
- every one of those fixtures reconciles exactly with Database v4;
- the other 18 racecourses contain 61 BHA fixtures and 549 races;
- all 549 races are absent from Database v4.

The June 2020 defect therefore predates the Inside Rails database construction. Database v4 faithfully preserves the incomplete Source Version 1 population.

For June 2020, the source defect has now been characterised as the complete omission of 61 fixtures / 549 races across 18 racecourses, while the 29 retained fixtures / 260 races are complete.

In [32]:
# Exact July 2020 BHA race-population count using the established method.

july_fixture_url = (
    f"{BHA_API_ROOT}/bha/v1/fixtures/"
    "?fromdate=2020-07-01"
    "&todate=2020-07-31"
    "&resultsAvailable=1"
    "&order=asc"
    "&page=1"
    "&per_page=100"
)

first_page = get_bha_json(july_fixture_url)

july_fixtures = list(first_page["data"])

for page in range(2, int(first_page["last_page"]) + 1):
    page_url = (
        f"{BHA_API_ROOT}/bha/v1/fixtures/"
        "?fromdate=2020-07-01"
        "&todate=2020-07-31"
        "&resultsAvailable=1"
        "&order=asc"
        f"&page={page}"
        "&per_page=100"
    )
    page_payload = get_bha_json(page_url)
    july_fixtures.extend(page_payload["data"])


july_active_fixtures = [
    fixture
    for fixture in july_fixtures
    if int(fixture.get("abandonedReasonCode") or 0) == 0
]

july_abandoned_fixtures = [
    fixture
    for fixture in july_fixtures
    if int(fixture.get("abandonedReasonCode") or 0) != 0
]


july_exact_rows = []

for fixture in july_active_fixtures:
    race_url = (
        f"{BHA_API_ROOT}/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/{fixture['fixtureId']}/races"
    )

    race_payload = get_bha_json(race_url)
    races = race_payload["data"]

    july_exact_rows.append(
        {
            "fixture_date": pd.to_datetime(
                fixture["fixtureDate"]
            ).strftime("%Y-%m-%d"),
            "course_name": fixture["courseName"],
            "fixture_id": fixture["fixtureId"],
            "fixture_year": fixture["fixtureYear"],
            "actual_races": len(races),
        }
    )

    time.sleep(1.5)


july_exact = pd.DataFrame(july_exact_rows)

bha_july_races = int(
    july_exact["actual_races"].sum()
)

with connect_read_only(DATABASE_V4) as conn:
    v4_july_races = conn.execute(
        """
        SELECT COUNT(*)
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-07-01'
          AND raw_date < '2020-08-01'
        """
    ).fetchone()[0]


print("BHA fixture records:", len(july_fixtures))
print("BHA active fixtures:", len(july_active_fixtures))
print("BHA wholly abandoned fixtures:", len(july_abandoned_fixtures))
print("BHA completed-fixture race-list records:", bha_july_races)
print("Database v4 races:", v4_july_races)
print("Exact July shortfall:", bha_july_races - v4_july_races)

BHA fixture records: 181
BHA active fixtures: 106
BHA wholly abandoned fixtures: 75
BHA completed-fixture race-list records: 915
Database v4 races: 275
Exact July shortfall: 640


### Finding — July 2020 has an exact 640-race population shortfall

The corrected BHA reconciliation method was applied across July 2020.

The BHA fixture population contains:

- 181 fixture records;
- 106 non-abandoned fixtures;
- 75 wholly abandoned fixtures.

After excluding wholly abandoned fixtures, the BHA race-list endpoints contain **915 race records** attached to active fixtures.

Database v4 contains **275 source-backed Great Britain race occurrences** for July 2020.

The exact July population shortfall is therefore:

**915 - 275 = 640 races**

This exactly matches the earlier fixture-level screening shortfall of 640 races.

As with June, the July deficit is therefore not an artefact of abandoned fixtures or the initial screening approximation.

The next step is to test whether July shows the same whole-racecourse / whole-fixture omission pattern established for June.

In [33]:
# Localise the exact July 2020 population deficit by racecourse.

bha_july_by_course = (
    july_exact
    .groupby("course_name", as_index=False)
    .agg(
        bha_fixtures=("fixture_id", "nunique"),
        bha_races=("actual_races", "sum"),
    )
    .sort_values(
        ["bha_races", "course_name"],
        ascending=[False, True],
    )
)

with connect_read_only(DATABASE_V4) as conn:
    v4_july_by_course = pd.read_sql_query(
        """
        SELECT
            governed_racecourse_name,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-07-01'
          AND raw_date < '2020-08-01'
        GROUP BY governed_racecourse_name
        ORDER BY v4_races DESC, governed_racecourse_name
        """,
        conn,
    )

print("BHA completed-fixture population — July 2020")
display(bha_july_by_course)

print("\nDatabase v4 population — July 2020")
display(v4_july_by_course)

BHA completed-fixture population — July 2020


,course_name,bha_fixtures,bha_races
14,Haydock Park,5,45
36,York,5,44
1,Ayr,5,42
3,Bath,5,42
22,Newmarket,5,41
30,Southwell,4,36
34,Windsor,4,36
35,Wolverhampton,4,35
12,Great Yarmouth,4,34
8,Chepstow,4,33



Database v4 population — July 2020


,governed_racecourse_name,v4_races
0,York,44
1,Newmarket — July Course,41
2,Great Yarmouth,34
3,Chepstow,33
4,Catterick,27
5,Kempton Park,27
6,Musselburgh,25
7,Doncaster,19
8,Chelmsford City,9
9,Newcastle,9


### Finding — July 2020 repeats the whole-racecourse omission pattern

The July racecourse-level comparison reproduces the same structured defect observed in June.

The BHA completed-fixture population contains **915 races across 37 racecourses**.

Database v4 contains **275 races across 11 governed racecourses**.

After applying the established racecourse identity mappings:

- York — BHA 44 / v4 44
- Newmarket — July Course — BHA 41 / v4 41
- Great Yarmouth — BHA 34 / v4 34
- Chepstow — BHA 33 / v4 33
- Catterick — BHA 27 / v4 27
- Kempton Park — BHA 27 / v4 27
- Musselburgh — BHA 25 / v4 25
- Doncaster — BHA 19 / v4 19
- Chelmsford City — BHA 9 / v4 9
- Newcastle — BHA 9 / v4 9
- Epsom Downs — BHA 7 / v4 7

Those 11 retained racecourses total exactly **275 races**.

The remaining BHA racecourse populations total **640 races**, exactly equal to the complete July shortfall.

July therefore appears to repeat June's pattern: complete populations are retained for a subset of racecourses while the other racecourse populations are wholly absent.

The next step is to verify this at individual fixture level, including the governed Newmarket and Catterick name mappings.

In [34]:
# Verify whether every retained July fixture reconciles individually.

july_course_map = {
    "Newmarket": "Newmarket — July Course",
    "Catterick Bridge": "Catterick",
}

july_exact_mapped = july_exact.copy()

july_exact_mapped["governed_racecourse_name"] = (
    july_exact_mapped["course_name"]
    .replace(july_course_map)
)

retained_july_courses = set(
    v4_july_by_course["governed_racecourse_name"]
)

bha_retained_july = (
    july_exact_mapped[
        july_exact_mapped["governed_racecourse_name"].isin(
            retained_july_courses
        )
    ]
    [
        [
            "fixture_date",
            "course_name",
            "governed_racecourse_name",
            "fixture_id",
            "actual_races",
        ]
    ]
    .rename(columns={"actual_races": "bha_races"})
)

with connect_read_only(DATABASE_V4) as conn:
    v4_july_by_date_course = pd.read_sql_query(
        """
        SELECT
            raw_date AS fixture_date,
            governed_racecourse_name,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-07-01'
          AND raw_date < '2020-08-01'
        GROUP BY
            raw_date,
            governed_racecourse_name
        ORDER BY
            raw_date,
            governed_racecourse_name
        """,
        conn,
    )

retained_july_check = (
    bha_retained_july
    .merge(
        v4_july_by_date_course,
        on=[
            "fixture_date",
            "governed_racecourse_name",
        ],
        how="outer",
    )
)

retained_july_check["bha_races"] = (
    retained_july_check["bha_races"]
    .fillna(0)
    .astype(int)
)

retained_july_check["v4_races"] = (
    retained_july_check["v4_races"]
    .fillna(0)
    .astype(int)
)

retained_july_check["difference"] = (
    retained_july_check["bha_races"]
    - retained_july_check["v4_races"]
)

july_fixture_mismatches = retained_july_check[
    retained_july_check["difference"] != 0
]

print(
    "Retained BHA fixtures:",
    len(bha_retained_july),
)

print(
    "Retained BHA races:",
    int(bha_retained_july["bha_races"].sum()),
)

print(
    "Fixture/date-course mismatches:",
    len(july_fixture_mismatches),
)

display(july_fixture_mismatches)

print("\nCompletely absent July racecourses:")

bha_absent_july_courses = (
    july_exact_mapped[
        ~july_exact_mapped["governed_racecourse_name"].isin(
            retained_july_courses
        )
    ]
    .groupby(
        ["course_name", "governed_racecourse_name"],
        as_index=False,
    )
    .agg(
        bha_fixtures=("fixture_id", "nunique"),
        bha_races=("actual_races", "sum"),
    )
    .sort_values(
        ["bha_races", "course_name"],
        ascending=[False, True],
    )
)

display(bha_absent_july_courses)

print(
    "Absent racecourses:",
    bha_absent_july_courses[
        "governed_racecourse_name"
    ].nunique(),
)

print(
    "Absent BHA races:",
    int(bha_absent_july_courses["bha_races"].sum()),
)

Retained BHA fixtures: 32
Retained BHA races: 275
Fixture/date-course mismatches: 0


,fixture_date,course_name,governed_racecourse_name,fixture_id,bha_races,v4_races,difference



Completely absent July racecourses:


,course_name,governed_racecourse_name,bha_fixtures,bha_races
8,Haydock Park,Haydock Park,5,45
1,Ayr,Ayr,5,42
3,Bath,Bath,5,42
20,Southwell,Southwell,4,36
24,Windsor,Windsor,4,36
25,Wolverhampton,Wolverhampton,4,35
19,Sandown Park,Sandown Park,4,33
6,Goodwood,Goodwood,4,31
23,Uttoxeter,Uttoxeter,3,27
11,Market Rasen,Market Rasen,3,26


Absent racecourses: 26
Absent BHA races: 640


### Finding — July retained fixtures are complete and 26 racecourse populations are wholly absent

The apparent July racecourse-level pattern was tested at individual fixture level using the governed racecourse mappings, including:

- `Newmarket` → `Newmarket — July Course`;
- `Catterick Bridge` → `Catterick`.

Results:

- retained BHA fixtures: **32**;
- retained BHA races: **275**;
- Database v4 races across those fixtures: **275**;
- fixture/date-course mismatches: **0**.

Every retained July fixture therefore reconciles exactly.

The other **26 racecourses are completely absent from Database v4** for July 2020. Together they account for:

- **74 BHA fixtures**;
- **640 BHA races**;
- **0 Database v4 races**.

Those 640 races exactly equal the complete July population shortfall.

July therefore reproduces the June 2020 defect structure: Source Version 1 / Database v4 retains complete meetings for a subset of racecourses while entirely omitting the remaining active fixture populations. There is no evidence of partial race loss within the fixtures that survive.

In [35]:
# Verify directly which governed GB racecourses have any admitted
# Source Version 1 runner records in July 2020.
#
# No use of the prohibited Source V1 `off` field.

with connect_read_only(DATABASE_V4) as conn:
    source_v1_july_course_presence = pd.read_sql_query(
        """
        SELECT
            ref.racecourse_name AS governed_racecourse_name,
            ref.candidate_course_label AS source_course_label,
            COUNT(raw.source_record_id) AS source_runner_rows,
            COUNT(DISTINCT raw.date) AS source_dates
        FROM view_gb_racecourse_identity_reference AS ref
        JOIN source_raceform_v1_record AS raw
          ON raw.course = ref.candidate_course_label
         AND raw.structural_status = 'admitted_runner_record'
         AND raw.date >= '2020-07-01'
         AND raw.date < '2020-08-01'
        GROUP BY
            ref.racecourse_name,
            ref.candidate_course_label
        ORDER BY
            ref.racecourse_name,
            ref.candidate_course_label
        """,
        conn,
    )

display(source_v1_july_course_presence)

print(
    "Governed GB racecourses represented in raw Source V1:",
    source_v1_july_course_presence[
        "governed_racecourse_name"
    ].nunique(),
)

print(
    "GB Source V1 runner rows in July:",
    int(
        source_v1_july_course_presence[
            "source_runner_rows"
        ].sum()
    ),
)

,governed_racecourse_name,source_course_label,source_runner_rows,source_dates
0,Catterick,Catterick,265,3
1,Chelmsford City,Chelmsford (AW),94,1
2,Chepstow,Chepstow,314,4
3,Doncaster,Doncaster,211,2
4,Epsom Downs,Epsom,68,1
5,Great Yarmouth,Yarmouth,292,4
6,Kempton Park,Kempton (AW),292,3
7,Musselburgh,Musselburgh,204,3
8,Newcastle,Newcastle (AW),88,1
9,Newmarket — July Course,Newmarket (July),368,5


Governed GB racecourses represented in raw Source V1: 11
GB Source V1 runner rows in July: 2666


### Finding — July 2020 defect originates in Source Version 1

The immutable Source Version 1 raw evidence was inspected directly for July 2020 without using the prohibited `off` field.

Admitted Source Version 1 runner records exist for exactly **11 governed British racecourses**:

- Catterick
- Chelmsford City
- Chepstow
- Doncaster
- Epsom Downs
- Great Yarmouth
- Kempton Park
- Musselburgh
- Newcastle
- Newmarket — July Course
- York

These are exactly the same 11 racecourses represented in Database v4.

Earlier reconciliation established that those racecourses account for:

- **32 BHA fixtures**;
- **275 BHA races**;
- **275 Database v4 races**;
- **0 fixture-level mismatches**.

The other **26 racecourses**, containing **74 fixtures and 640 BHA races**, have no July population in Database v4 and no admitted July runner records in the immutable Source Version 1 raw evidence.

The July population defect therefore originates in Source Version 1 rather than in later Inside Rails database construction.

### June–July combined result

June and July 2020 together account for:

- **135 wholly omitted BHA fixtures**;
- **1,189 BHA races absent from Source Version 1**.

For every retained fixture tested across both months, the BHA race count reconciles exactly with Database v4.

The evidence therefore indicates selective omission of complete fixture populations rather than partial loss of races within retained meetings.

In [36]:
# Exact January 2020 BHA race-population count using the established method.

january_fixture_url = (
    f"{BHA_API_ROOT}/bha/v1/fixtures/"
    "?fromdate=2020-01-01"
    "&todate=2020-01-31"
    "&resultsAvailable=1"
    "&order=asc"
    "&page=1"
    "&per_page=100"
)

first_page = get_bha_json(january_fixture_url)

january_fixtures = list(first_page["data"])

for page in range(2, int(first_page["last_page"]) + 1):
    page_url = (
        f"{BHA_API_ROOT}/bha/v1/fixtures/"
        "?fromdate=2020-01-01"
        "&todate=2020-01-31"
        "&resultsAvailable=1"
        "&order=asc"
        f"&page={page}"
        "&per_page=100"
    )
    page_payload = get_bha_json(page_url)
    january_fixtures.extend(page_payload["data"])


january_active_fixtures = [
    fixture
    for fixture in january_fixtures
    if int(fixture.get("abandonedReasonCode") or 0) == 0
]

january_abandoned_fixtures = [
    fixture
    for fixture in january_fixtures
    if int(fixture.get("abandonedReasonCode") or 0) != 0
]


january_exact_rows = []

for fixture in january_active_fixtures:
    race_url = (
        f"{BHA_API_ROOT}/bha/v1/fixtures/"
        f"{fixture['fixtureYear']}/{fixture['fixtureId']}/races"
    )

    race_payload = get_bha_json(race_url)
    races = race_payload["data"]

    january_exact_rows.append(
        {
            "fixture_date": pd.to_datetime(
                fixture["fixtureDate"]
            ).strftime("%Y-%m-%d"),
            "course_name": fixture["courseName"],
            "fixture_id": fixture["fixtureId"],
            "fixture_year": fixture["fixtureYear"],
            "actual_races": len(races),
        }
    )

    time.sleep(1.5)


january_exact = pd.DataFrame(january_exact_rows)

bha_january_races = int(
    january_exact["actual_races"].sum()
)

with connect_read_only(DATABASE_V4) as conn:
    v4_january_races = conn.execute(
        """
        SELECT COUNT(*)
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-01-01'
          AND raw_date < '2020-02-01'
        """
    ).fetchone()[0]


print("BHA fixture records:", len(january_fixtures))
print("BHA active fixtures:", len(january_active_fixtures))
print("BHA wholly abandoned fixtures:", len(january_abandoned_fixtures))
print("BHA completed-fixture race-list records:", bha_january_races)
print("Database v4 races:", v4_january_races)
print("Exact January shortfall:", bha_january_races - v4_january_races)

BHA fixture records: 112
BHA active fixtures: 104
BHA wholly abandoned fixtures: 8
BHA completed-fixture race-list records: 731
Database v4 races: 584
Exact January shortfall: 147


### Finding — January 2020 has an exact 147-race population shortfall

The corrected BHA reconciliation method was applied across January 2020.

The BHA fixture population contains:

- 112 fixture records;
- 104 non-abandoned fixtures;
- 8 wholly abandoned fixtures.

After excluding wholly abandoned fixtures and enumerating the race-list records for the remaining fixtures, the BHA population is **731 races**.

Database v4 contains **584 source-backed Great Britain race occurrences** for January 2020.

The exact January shortfall is therefore:

**731 - 584 = 147 races**

The earlier fixture-level screening method estimated 736 races and a 152-race shortfall. It therefore over-counted the January completed-race population by **5 races**.

This confirms the screening method is useful for localisation but cannot be treated as an exact population count. Exact reconciliation requires race-list enumeration after wholly abandoned fixtures have been excluded.

In [37]:
# Localise the exact January 2020 population deficit by racecourse.

bha_january_by_course = (
    january_exact
    .groupby("course_name", as_index=False)
    .agg(
        bha_fixtures=("fixture_id", "nunique"),
        bha_races=("actual_races", "sum"),
    )
    .sort_values(
        ["bha_races", "course_name"],
        ascending=[False, True],
    )
)

with connect_read_only(DATABASE_V4) as conn:
    v4_january_by_course = pd.read_sql_query(
        """
        SELECT
            governed_racecourse_name,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-01-01'
          AND raw_date < '2020-02-01'
        GROUP BY governed_racecourse_name
        ORDER BY v4_races DESC, governed_racecourse_name
        """,
        conn,
    )

print("BHA completed-fixture population — January 2020")
display(bha_january_by_course)

print("\nDatabase v4 population — January 2020")
display(v4_january_by_course)

BHA completed-fixture population — January 2020


,course_name,bha_fixtures,bha_races
17,Lingfield Park,12,90
22,Newcastle,10,72
26,Southwell,10,70
32,Wolverhampton,9,65
15,Kempton Park,7,55
3,Chelmsford City,6,41
2,Catterick Bridge,4,27
1,Ayr,3,20
5,Chepstow,3,20
6,Doncaster,3,20



Database v4 population — January 2020


,governed_racecourse_name,v4_races
0,Lingfield Park,90
1,Newcastle,72
2,Southwell,70
3,Kempton Park,55
4,Chelmsford City,41
5,Catterick,27
6,Ayr,20
7,Chepstow,20
8,Doncaster,20
9,Musselburgh,19


### Finding — January 2020 also exhibits whole-racecourse population omission

The January racecourse-level comparison reveals the same structured population defect found in June and July.

The exact BHA completed-fixture population is **731 races**. Database v4 contains **584 races**.

Every BHA racecourse represented in Database v4 has the same monthly race total in both sources, allowing for the established mapping:

- `Catterick Bridge` → `Catterick`.

Eight BHA racecourses are completely absent from Database v4 for January 2020:

- Wolverhampton — 65 races
- Sedgefield — 14 races
- Taunton — 14 races
- Warwick — 14 races
- Wincanton — 14 races
- Wetherby — 13 races
- Sandown Park — 7 races
- Uttoxeter — 6 races

These eight racecourses contain **147 BHA races**, exactly equal to the complete January shortfall.

This is important because January predates the March 2020 interruption to British racing. The selective whole-racecourse omission pattern therefore cannot be explained solely as a consequence of racing's COVID-19 shutdown and restart.

The next step is to confirm that the retained January population also reconciles at individual fixture level.

In [38]:
# Verify January 2020 retained fixtures individually.

january_course_map = {
    "Catterick Bridge": "Catterick",
}

january_exact_mapped = january_exact.copy()

january_exact_mapped["governed_racecourse_name"] = (
    january_exact_mapped["course_name"]
    .replace(january_course_map)
)

retained_january_courses = set(
    v4_january_by_course["governed_racecourse_name"]
)

bha_retained_january = (
    january_exact_mapped[
        january_exact_mapped[
            "governed_racecourse_name"
        ].isin(retained_january_courses)
    ]
    [
        [
            "fixture_date",
            "course_name",
            "governed_racecourse_name",
            "fixture_id",
            "actual_races",
        ]
    ]
    .rename(columns={"actual_races": "bha_races"})
)

with connect_read_only(DATABASE_V4) as conn:
    v4_january_by_date_course = pd.read_sql_query(
        """
        SELECT
            raw_date AS fixture_date,
            governed_racecourse_name,
            COUNT(*) AS v4_races
        FROM view_gb_reconciled_race_occurrences_with_racecourse
        WHERE raw_date >= '2020-01-01'
          AND raw_date < '2020-02-01'
        GROUP BY
            raw_date,
            governed_racecourse_name
        ORDER BY
            raw_date,
            governed_racecourse_name
        """,
        conn,
    )

retained_january_check = (
    bha_retained_january
    .merge(
        v4_january_by_date_course,
        on=[
            "fixture_date",
            "governed_racecourse_name",
        ],
        how="outer",
    )
)

retained_january_check["bha_races"] = (
    retained_january_check["bha_races"]
    .fillna(0)
    .astype(int)
)

retained_january_check["v4_races"] = (
    retained_january_check["v4_races"]
    .fillna(0)
    .astype(int)
)

retained_january_check["difference"] = (
    retained_january_check["bha_races"]
    - retained_january_check["v4_races"]
)

january_fixture_mismatches = retained_january_check[
    retained_january_check["difference"] != 0
]

print(
    "Retained BHA fixtures:",
    len(bha_retained_january),
)

print(
    "Retained BHA races:",
    int(bha_retained_january["bha_races"].sum()),
)

print(
    "Fixture/date-course mismatches:",
    len(january_fixture_mismatches),
)

display(january_fixture_mismatches)

print("\nCompletely absent January racecourses:")

bha_absent_january_courses = (
    january_exact_mapped[
        ~january_exact_mapped[
            "governed_racecourse_name"
        ].isin(retained_january_courses)
    ]
    .groupby(
        ["course_name", "governed_racecourse_name"],
        as_index=False,
    )
    .agg(
        bha_fixtures=("fixture_id", "nunique"),
        bha_races=("actual_races", "sum"),
    )
    .sort_values(
        ["bha_races", "course_name"],
        ascending=[False, True],
    )
)

display(bha_absent_january_courses)

print(
    "Absent racecourses:",
    bha_absent_january_courses[
        "governed_racecourse_name"
    ].nunique(),
)

print(
    "Absent BHA fixtures:",
    int(bha_absent_january_courses["bha_fixtures"].sum()),
)

print(
    "Absent BHA races:",
    int(bha_absent_january_courses["bha_races"].sum()),
)

Retained BHA fixtures: 83
Retained BHA races: 584
Fixture/date-course mismatches: 0


,fixture_date,course_name,governed_racecourse_name,fixture_id,bha_races,v4_races,difference



Completely absent January racecourses:


,course_name,governed_racecourse_name,bha_fixtures,bha_races
7,Wolverhampton,Wolverhampton,9,65
1,Sedgefield,Sedgefield,2,14
2,Taunton,Taunton,2,14
4,Warwick,Warwick,2,14
6,Wincanton,Wincanton,2,14
5,Wetherby,Wetherby,2,13
0,Sandown Park,Sandown Park,1,7
3,Uttoxeter,Uttoxeter,1,6


Absent racecourses: 8
Absent BHA fixtures: 21
Absent BHA races: 147


### Finding — January retained fixtures are complete and 21 fixtures are wholly absent

The January 2020 retained population was tested at individual fixture level.

Results:

- retained BHA fixtures: **83**;
- retained BHA races: **584**;
- Database v4 races across those fixtures: **584**;
- fixture/date-course mismatches: **0**.

Every retained January fixture therefore reconciles exactly.

Eight racecourses are completely absent:

- Wolverhampton — 9 fixtures / 65 races
- Sedgefield — 2 / 14
- Taunton — 2 / 14
- Warwick — 2 / 14
- Wincanton — 2 / 14
- Wetherby — 2 / 13
- Sandown Park — 1 / 7
- Uttoxeter — 1 / 6

Together these account for **21 fixtures and 147 races**, exactly equal to the January population shortfall.

January therefore exhibits the same structural defect as June and July: complete retained fixtures alongside complete omission of other fixture populations, with no partial race loss observed within retained meetings.

In [39]:
# Verify directly which governed GB racecourses have admitted
# Source Version 1 runner records in January 2020.
#
# No use of the prohibited Source V1 `off` field.

with connect_read_only(DATABASE_V4) as conn:
    source_v1_january_course_presence = pd.read_sql_query(
        """
        SELECT
            ref.racecourse_name AS governed_racecourse_name,
            ref.candidate_course_label AS source_course_label,
            COUNT(raw.source_record_id) AS source_runner_rows,
            COUNT(DISTINCT raw.date) AS source_dates
        FROM view_gb_racecourse_identity_reference AS ref
        JOIN source_raceform_v1_record AS raw
          ON raw.course = ref.candidate_course_label
         AND raw.structural_status = 'admitted_runner_record'
         AND raw.date >= '2020-01-01'
         AND raw.date < '2020-02-01'
        GROUP BY
            ref.racecourse_name,
            ref.candidate_course_label
        ORDER BY
            ref.racecourse_name,
            ref.candidate_course_label
        """,
        conn,
    )

display(source_v1_january_course_presence)

print(
    "Governed GB racecourses represented in raw Source V1:",
    source_v1_january_course_presence[
        "governed_racecourse_name"
    ].nunique(),
)

print(
    "GB Source V1 runner rows in January:",
    int(
        source_v1_january_course_presence[
            "source_runner_rows"
        ].sum()
    ),
)

,governed_racecourse_name,source_course_label,source_runner_rows,source_dates
0,Ascot,Ascot,45,1
1,Ayr,Ayr,155,3
2,Catterick,Catterick,220,4
3,Chelmsford City,Chelmsford (AW),365,6
4,Cheltenham,Cheltenham,129,2
5,Chepstow,Chepstow,175,3
6,Doncaster,Doncaster,208,3
7,Exeter,Exeter,141,2
8,Fakenham,Fakenham,105,2
9,Ffos Las,Ffos Las,51,1


Governed GB racecourses represented in raw Source V1: 25
GB Source V1 runner rows in January: 4963


### Finding — January 2020 defect originates in Source Version 1

The immutable Source Version 1 raw evidence was inspected directly for January 2020 without using the prohibited `off` field.

Admitted Source Version 1 runner records are present for **25 governed British racecourses**.

These are exactly the racecourses represented in Database v4.

The eight BHA racecourses previously identified as completely missing from Database v4 have no admitted January runner records in Source Version 1:

- Wolverhampton
- Sedgefield
- Taunton
- Warwick
- Wincanton
- Wetherby
- Sandown Park
- Uttoxeter

Those omitted racecourses account for:

- **21 BHA fixtures**
- **147 BHA races**

Earlier fixture-level reconciliation established that all **83 retained fixtures / 584 retained races** reconcile exactly between the BHA and Database v4.

The January 2020 population defect therefore originates in Source Version 1 rather than in later Inside Rails database construction.

January, June and July now independently exhibit the same source-defect structure: complete retained fixtures alongside complete omission of other fixture populations.

## Audit closeout — Source Version 1 is not a reliable authority for the 2020 GB race population

### Question

This audit began after a known omission was identified on 6 June 2020: official BHA records showed 28 British races, while Source Version 1 / Database v4 contained only the 10 Newcastle races.

The audit therefore asked whether Great Britain races that officially occurred were missing from the source population inherited by Database v4.

### What the audit established

At annual level:

- official BHA 2020 races: **7,874**
- Database v4 GB races: **6,287**
- annual difference: **1,587 races**

Control years were much closer:

- 2019: BHA 10,087 / v4 10,085
- 2021: BHA 10,354 / v4 10,353

The 2020 difference is therefore anomalously large.

### Exact month investigations

Three materially deficient months were investigated using BHA fixture status and fixture race-list records.

| Month | BHA races | Database v4 | Shortfall |
|---|---:|---:|---:|
| January | 731 | 584 | **147** |
| June | 809 | 260 | **549** |
| July | 915 | 275 | **640** |

These three months alone establish **1,336 races absent from Source Version 1 / Database v4**.

A September control was also examined exactly:

- BHA completed-fixture race population: **953**
- Database v4: **953**
- difference: **0**

September therefore demonstrates that the reconciliation method can produce an exact match where Source Version 1 is complete.

### Nature of the defect

The missing population is highly structured.

For January:

- 83 retained BHA fixtures / 584 races;
- every retained fixture reconciled exactly;
- 21 other fixtures / 147 races were wholly absent.

For June:

- 29 retained BHA fixtures / 260 races;
- every retained fixture reconciled exactly;
- 61 other fixtures / 549 races were wholly absent.

For July:

- 32 retained BHA fixtures / 275 races;
- every retained fixture reconciled exactly;
- 74 other fixtures / 640 races were wholly absent.

Across the three investigated deficient months:

- **144 retained fixtures** reconcile exactly;
- **1,119 retained races** reconcile exactly;
- **156 complete fixtures** are absent;
- those absent fixtures contain **1,336 races**.

No partial race loss was observed inside any retained fixture.

The defect therefore behaves as selective omission of complete fixture populations rather than corruption or random loss of individual races within retained meetings.

### Provenance of the defect

The immutable raw Source Version 1 mirror was inspected directly without using the prohibited Source Version 1 `off` field.

For January, June and July, raw Source Version 1 runner records existed for exactly the same racecourses represented in Database v4. Racecourses whose BHA fixtures were absent from Database v4 also had no corresponding runner population in Source Version 1.

The population defect therefore **originates in Source Version 1**.

Database v4 did not introduce these omissions; it faithfully inherited the incomplete source population.

### BHA race-list caveat

The BHA fixture race-list endpoint is not by itself an unconditional list of races that ran.

A September Worcester fixture retained a programmed race-list record despite the fixture being officially:

- `abandonedReasonCode = 1`;
- `resultsAvailable = 0`;
- `ABANDONED - Abandoned (72 Hours Before)`.

Exact population reconciliation must therefore account for fixture abandonment status before treating fixture race-list records as completed-race population.

### Audit conclusion

It is unnecessary to enumerate every remaining 2020 month individually before making the source-governance decision.

The evidence is sufficient to conclude that:

> **Source Version 1 is not a defensible authoritative source for the complete Great Britain race population in 2020.**

This does **not** imply that every month in 2020 is incomplete; September was proved complete. It means that Source Version 1 cannot safely determine whether a British race existed merely from presence or absence in that source.

Source Version 1 remains valuable immutable third-party evidence and may continue to supply useful race and runner attributes. It should not, however, remain the population authority for British racing where a suitable official source is available.

Database v4 remains immutable and valid as a governed representation of Source Version 1. This audit does not patch or mutate Database v4.

### Decision

Stop the month-by-month 2020 completeness audit here.

The next research priority is the BHA official-source feasibility study:

> **How far back does the BHA provide usable official fixture, race and result data, and which authoritative population/result capabilities are available at each historical depth?**

No Database v5 architecture should be designed until that source capability and historical boundary are established.

If the BHA provides sufficient historical coverage, the likely conceptual direction is to make the official BHA population authoritative for Great Britain and reconcile Source Version 1 onto that official population as secondary/enrichment evidence rather than using Source Version 1 to define which races existed.